
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Lab - Building Reproducible AI Agents with MLflow Tracing

## Overview

In this hands-on lab, you'll apply the concepts learned in the previous demonstrations to build, trace, and register your own AI agent with Unity Catalog. You'll implement custom tracing functions, validate agent behavior, and register your agent to Unity Catalog for production use. This lab focuses on practical implementation skills that are essential for building robust, observable AI systems in enterprise environments.

## Learning Objectives

By the end of this lab, you will be able to:

- Implement custom tracing functions with proper validation and error handling
- Apply MLflow tagging strategies to organize and categorize agent traces
- Create tool validation functions to ensure agents use appropriate tools
- Log and register AI agents to Unity Catalog with complete dependency management
- Log, register, and inference agents from both MLflow and Unity Catalog registries

## A. Environment Setup

### A1. Compute Requirements

**🚨 REQUIRED - SELECT SERVERLESS COMPUTE**

This course has been configured to run on Serverless compute. While classic compute may also work, testing has been performed on serverless.

**This demo was tested using version 5 of Serverless compute.** To ensure that you are using the correct version of Serverless, please [see this documentation on viewing and changing your notebook's Serverless verison.](https://docs.databricks.com/aws/en/compute/serverless/dependencies) 

### A2. Install Dependencies

Install the required Python libraries for MLflow tracing and agent functionality.

In [0]:
%run ../Includes/Classroom-Setup-3.3

### A3. Inspect the Airbnb Dataset 
As a part of the classroom setup, the Airbnb dataset has been processed and stored as a Delta table within Unity Catalog. Run the next cell to query the first few rows of the dataset.

In [0]:
df = spark.read.table('sf_airbnb_listings')
display(df.limit(5))

### A4. Load the Agent

Import the pre-configured agent that you'll be working with throughout this lab.

**NOTE:** `mlflow.autolog` has been configured as a part of the agent's code, so we do not need to initiate it in this notebook.

We'll start by using a custom function that will build the `demo_agent_config.json` file. This needs to be specific to the **catalog** and **schema** you defined above. In practice, making this dynamic or static depends on your use case.

In [0]:
# Reload the agent module
%reload_ext autoreload
%autoreload 2

from lab_agent import AGENT as agent

## B. Custom Tracing with Tags

In this section, you'll implement custom tracing functions with MLflow tags to organize and categorize your traces.

### B1. Define Trace Tags

Create a tags dictionary that will help categorize and organize your traces. These tags will be applied to your custom tracing functions.

In [0]:
## Create a tags dictionary with the following keys and values:
## - component: "input_validation"
## - stage: "preprocessing" 
## - span_scope: "tool_function"
## - env: "dev"
## - trace_version: "v1.0.0"
## - input_type: "question"

tags = {
    <FILL_IN>
}

##### Task B1.1: Define Trace Tags ANSWER
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
tags = {
    "component": "input_validation",
    "stage": "preprocessing",
    "span_scope": "tool_function",
    "env": "dev",
    "trace_version": "v1.0.0",
    "input_type": "question"
}
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
const el = document.getElementById("copy-block");
if (!el) return;

const text = el.innerText;

// Preferred modern API
if (navigator.clipboard && navigator.clipboard.writeText) {
navigator.clipboard.writeText(text)
.then(() => alert("Copied to clipboard"))
.catch(err => {
console.error("Clipboard write failed:", err);
fallbackCopy(text);
});
} else {
fallbackCopy(text);
}
}

function fallbackCopy(text) {
const textarea = document.createElement("textarea");
textarea.value = text;
textarea.style.position = "fixed";
textarea.style.left = "-9999px";
document.body.appendChild(textarea);
textarea.select();
try {
document.execCommand("copy");
alert("Copied to clipboard");
} catch (err) {
console.error("Fallback copy failed:", err);
alert("Could not copy to clipboard. Please copy manually.");
} finally {
document.body.removeChild(textarea);
}
}
</script>
</details>

### B2. Import Required Libraries

Import the necessary MLflow libraries for custom tracing implementation. Specifically, bring in the module `SpanType` from `mlflow.entities`.

In [0]:
import mlflow 
from mlflow.entities import SpanType

### B3. Create Tool Validation Functions

You'll create two helper agent tools (Python functions) that validate whether the agent used tools in its response. This is useful for ensuring your agent follows expected behavior patterns. This section will mimic what `lab_agent_update.py` is doing - but will focus on the fundamentals of custom tracing and testing your custom traces with tags. The step for integrating the code below into your agent has been completed for you and you can always view `lab_agent_update.py` for the full integration.

The code present in this section is mostly complete, allowing you to focus on the MLflow aspect rather than the specific use case. However, feel free to add your own custom code as needed.

#### B3.1 Tool Usage Validation Function

This function checks whether the model response used a tool and returns structured results. Note most of the custom code has been created for you.

In [0]:
## Complete the validate_tool_usage function
## Use the @mlflow.trace decorator with span_type=SpanType.TOOL and name="Check Tool Usage"

<FILL_IN>
def validate_tool_usage(result):
    """Check whether the model response used a tool and return a structured result."""
    
    def _get(obj, key, default=None):
        if isinstance(obj, dict):
            return obj.get(key, default)
        return getattr(obj, key, default) if hasattr(obj, key) else default

    # Extract outputs from result
    output_list = _get(result, "output", []) or []

    # Find tool usage
    tool_calls = [
        item for item in output_list
        if _get(item, "type") in ("function_call", "function_call_output")
    ]

    if not tool_calls:
        return {
            "used_tool": False,
            "error": "No tools were used during the model response.",
        }

    # Collect tool names and call IDs for debugging
    tools_info = [
        {
            "name": _get(item, "name"),
            "call_id": _get(item, "call_id"),
            "type": _get(item, "type"),
        }
        for item in tool_calls
    ]

    return {
        "used_tool": True,
        "tools": tools_info,
        "tool_count": len(tools_info)
    }

##### Task B3.1: Tool Usage Validation Function ANSWER
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
@mlflow.trace(
    span_type=SpanType.TOOL,
    name="Check Tool Usage"
)
def validate_tool_usage(result):
    """Check whether the model response used a tool and return a structured result."""
    
    def _get(obj, key, default=None):
        if isinstance(obj, dict):
            return obj.get(key, default)
        return getattr(obj, key, default) if hasattr(obj, key) else default

    # Extract outputs from result
    output_list = _get(result, "output", []) or []

    # Find tool usage
    tool_calls = [
        item for item in output_list
        if _get(item, "type") in ("function_call", "function_call_output")
    ]

    if not tool_calls:
        return {
            "used_tool": False,
            "error": "No tools were used during the model response.",
        }

    # Collect tool names and call IDs for debugging
    tools_info = [
        {
            "name": _get(item, "name"),
            "call_id": _get(item, "call_id"),
            "type": _get(item, "type"),
        }
        for item in tool_calls
    ]

    return {
        "used_tool": True,
        "tools": tools_info,
        "tool_count": len(tools_info)
    }
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
const el = document.getElementById("copy-block");
if (!el) return;

const text = el.innerText;

// Preferred modern API
if (navigator.clipboard && navigator.clipboard.writeText) {
navigator.clipboard.writeText(text)
.then(() => alert("Copied to clipboard"))
.catch(err => {
console.error("Clipboard write failed:", err);
fallbackCopy(text);
});
} else {
fallbackCopy(text);
}
}

function fallbackCopy(text) {
const textarea = document.createElement("textarea");
textarea.value = text;
textarea.style.position = "fixed";
textarea.style.left = "-9999px";
document.body.appendChild(textarea);
textarea.select();
try {
document.execCommand("copy");
alert("Copied to clipboard");
} catch (err) {
console.error("Fallback copy failed:", err);
alert("Could not copy to clipboard. Please copy manually.");
} finally {
document.body.removeChild(textarea);
}
}
</script>
</details>

#### B3.2 Response Evaluation Function

This function evaluates the model response and raises an error if no tool was used. Note most of the code has been completed for you.

In [0]:
## Complete the evaluate_response function
## Use the @mlflow.trace decorator with name="Evaluate Response"

<FILL_IN>
def evaluate_response(result):
    """Evaluate the model response and raise error if no tool was used."""
    
    validation = validate_tool_usage(result)
    
    if not validation["used_tool"]:
        # If no tool was used, explicitly raise an error
        raise ValueError(validation["error"])
    
    return {
        "message": f"{validation['tool_count']} tool(s) used successfully.",
        "details": validation["tools"]
    }

##### Task B3.2: Response Evaluation Function ANSWER
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
@mlflow.trace(name="Evaluate Response")
def evaluate_response(result):
    """Evaluate the model response and raise error if no tool was used."""
    
    validation = validate_tool_usage(result)
    
    if not validation["used_tool"]:
        # If no tool was used, explicitly raise an error
        raise ValueError(validation["error"])
    
    return {
        "message": f"{validation['tool_count']} tool(s) used successfully.",
        "details": validation["tools"]
    }
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
const el = document.getElementById("copy-block");
if (!el) return;

const text = el.innerText;

// Preferred modern API
if (navigator.clipboard && navigator.clipboard.writeText) {
navigator.clipboard.writeText(text)
.then(() => alert("Copied to clipboard"))
.catch(err => {
console.error("Clipboard write failed:", err);
fallbackCopy(text);
});
} else {
fallbackCopy(text);
}
}

function fallbackCopy(text) {
const textarea = document.createElement("textarea");
textarea.value = text;
textarea.style.position = "fixed";
textarea.style.left = "-9999px";
document.body.appendChild(textarea);
textarea.select();
try {
document.execCommand("copy");
alert("Copied to clipboard");
} catch (err) {
console.error("Fallback copy failed:", err);
alert("Could not copy to clipboard. Please copy manually.");
} finally {
document.body.removeChild(textarea);
}
}
</script>
</details>

### B4. Create LLM Call Function

Create a traced function that calls the agent and captures the interaction. Note this time you will need to build your function from scratch.

In [0]:
## Create a function called call_llm that:
## - Uses the @mlflow.trace decorator with name="Call LLM" and span_type=SpanType.CHAT_MODEL
## - Takes a question parameter
## - Returns agent.predict(question)

<FILL_IN>

##### Task B4.1: Create LLM Call Function ANSWER
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
@mlflow.trace(
    name="Call LLM",
    span_type=SpanType.CHAT_MODEL
)
def call_llm(question: str):
    return agent.predict(question)
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
const el = document.getElementById("copy-block");
if (!el) return;

const text = el.innerText;

// Preferred modern API
if (navigator.clipboard && navigator.clipboard.writeText) {
navigator.clipboard.writeText(text)
.then(() => alert("Copied to clipboard"))
.catch(err => {
console.error("Clipboard write failed:", err);
fallbackCopy(text);
});
} else {
fallbackCopy(text);
}
}

function fallbackCopy(text) {
const textarea = document.createElement("textarea");
textarea.value = text;
textarea.style.position = "fixed";
textarea.style.left = "-9999px";
document.body.appendChild(textarea);
textarea.select();
try {
document.execCommand("copy");
alert("Copied to clipboard");
} catch (err) {
console.error("Fallback copy failed:", err);
alert("Could not copy to clipboard. Please copy manually.");
} finally {
document.body.removeChild(textarea);
}
}
</script>
</details>

### B5. Create Main Processing Function

Create the main function that orchestrates the entire process, including tagging and validation. Note you will have to create this function from scratch with 3 main steps:
1. Update the current trace with new tags
2. Call the LLM using `call_llm()` you defined earlier
3. Get the tool evaluation using `evaluate_response()`

In [0]:
## Create a function called process_question that:
## - Uses the @mlflow.trace decorator with name="Process Question"
## - Takes user_input and include_metadata parameters
## - Updates the current trace with tags using mlflow.update_current_trace(tags)
## - Calls call_llm with the user_input
## - Evaluates the response using evaluate_response
## - Handles ValueError exceptions and prints appropriate messages

<FILL_IN>

##### Task B5.1: Create Main Processing Function ANSWER
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
@mlflow.trace(name="Process Question")
def process_question(user_input: str, include_metadata: bool = True):
    """Main function that calls LLM and formats response"""

    # Step 1: Update the current trace with new tags
    mlflow.update_current_trace(tags)
    
    # Step 2: Call the LLM
    llm_response = call_llm(user_input)
    
    # Step 3: Get tool evaluation
    try:
        summary = evaluate_response(llm_response)
        print(summary)
    except ValueError as e:
        print(f"Tool validation failed: {e}")
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
const el = document.getElementById("copy-block");
if (!el) return;

const text = el.innerText;

// Preferred modern API
if (navigator.clipboard && navigator.clipboard.writeText) {
navigator.clipboard.writeText(text)
.then(() => alert("Copied to clipboard"))
.catch(err => {
console.error("Clipboard write failed:", err);
fallbackCopy(text);
});
} else {
fallbackCopy(text);
}
}

function fallbackCopy(text) {
const textarea = document.createElement("textarea");
textarea.value = text;
textarea.style.position = "fixed";
textarea.style.left = "-9999px";
document.body.appendChild(textarea);
textarea.select();
try {
document.execCommand("copy");
alert("Copied to clipboard");
} catch (err) {
console.error("Fallback copy failed:", err);
alert("Could not copy to clipboard. Please copy manually.");
} finally {
document.body.removeChild(textarea);
}
}
</script>
</details>

## C. Testing Custom Traces

Now you will test your custom tracing implementation with different types of prompts.

### C1. Define Helper Functions and Test Prompts

The following creates a helper function to format prompts and define test cases. We also define what a success/fail prompt looks like for proper testing based on the logic you built up above.

In [0]:
def format_prompt(prompt: str) -> dict:
    return {
        "input": [
            {
                "role": "user",
                "content": prompt
            }
        ]
    }

In [0]:
# Define test prompts
success_prompt = "What is the price average for Mission?"
fail_prompt = "Hi!"

success_prompt_formatted = format_prompt(success_prompt)
fail_prompt_formatted = format_prompt(fail_prompt)

### C2. Test Successful Tool Usage

Test your tracing with a prompt that should trigger tool usage.

In [0]:
## Call process_question with success_prompt_formatted and tags
## This should result in successful tool usage

result = <FILL_IN>

##### Task C2.1: Test Successful Tool Usage ANSWER
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
result = process_question(success_prompt_formatted, tags)
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
const el = document.getElementById("copy-block");
if (!el) return;

const text = el.innerText;

// Preferred modern API
if (navigator.clipboard && navigator.clipboard.writeText) {
navigator.clipboard.writeText(text)
.then(() => alert("Copied to clipboard"))
.catch(err => {
console.error("Clipboard write failed:", err);
fallbackCopy(text);
});
} else {
fallbackCopy(text);
}
}

function fallbackCopy(text) {
const textarea = document.createElement("textarea");
textarea.value = text;
textarea.style.position = "fixed";
textarea.style.left = "-9999px";
document.body.appendChild(textarea);
textarea.select();
try {
document.execCommand("copy");
alert("Copied to clipboard");
} catch (err) {
console.error("Fallback copy failed:", err);
alert("Could not copy to clipboard. Please copy manually.");
} finally {
document.body.removeChild(textarea);
}
}
</script>
</details>

### C3. Test Failed Tool Usage

Test your tracing with a prompt that should not trigger tool usage.

In [0]:
## Call process_question with fail_prompt_formatted and tags
## This should result in a tool validation failure

result = <FILL_IN>

##### Task C3.1: Test Failed Tool Usage ANSWER
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
result = process_question(fail_prompt_formatted, tags)
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
const el = document.getElementById("copy-block");
if (!el) return;

const text = el.innerText;

// Preferred modern API
if (navigator.clipboard && navigator.clipboard.writeText) {
navigator.clipboard.writeText(text)
.then(() => alert("Copied to clipboard"))
.catch(err => {
console.error("Clipboard write failed:", err);
fallbackCopy(text);
});
} else {
fallbackCopy(text);
}
}

function fallbackCopy(text) {
const textarea = document.createElement("textarea");
textarea.value = text;
textarea.style.position = "fixed";
textarea.style.left = "-9999px";
document.body.appendChild(textarea);
textarea.select();
try {
document.execCommand("copy");
alert("Copied to clipboard");
} catch (err) {
console.error("Fallback copy failed:", err);
alert("Could not copy to clipboard. Please copy manually.");
} finally {
document.body.removeChild(textarea);
}
}
</script>
</details>

## D. Agent MLflow Logging and UC Registration 

In this section, you'll log your agent to MLflow and register it to Unity Catalog for production deployment. As a part of this classroom setup, there is a `.py` file named `lab_agent_update`. This file implements the custom tracing you filled out above but fit for using alongside `mlflow.types.responses`. Navigate to this file in the left side menu and scan the code for awareness before continuing.

### D1. Read Agent Configuration

The following prints a configuration file that defines your agent's settings and tools.

In [0]:
import json
# Read in Agent JSON config file 
with open('lab_agent_config.json', 'r') as f:
    agent_config = json.load(f)
print(agent_config)

### D2. Import Required Resources

Import the necessary MLflow resources for Unity Catalog integration. You will need to bring the proper libraries that will allow the model to access:
- Functions registered to Unity Catalog
- Tables registered to Unity Catalog
- Model serving endpoints hosted by Databricks

**NOTE:** For a review of the proper libraries, please see [this documentation](https://docs.databricks.com/aws/en/generative-ai/agent-framework/agent-authentication#implement-automatic-authentication-passthrough).

In [0]:
## Import the required classes from mlflow.models.resources

from importlib.metadata import version
from mlflow.models.resources import (
    <FILL_IN>
)

##### Task D2.1: Import Required Resources ANSWER
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
from importlib.metadata import version
from mlflow.models.resources import (
    DatabricksFunction,
    DatabricksTable,
    DatabricksServingEndpoint
)
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
const el = document.getElementById("copy-block");
if (!el) return;

const text = el.innerText;

// Preferred modern API
if (navigator.clipboard && navigator.clipboard.writeText) {
navigator.clipboard.writeText(text)
.then(() => alert("Copied to clipboard"))
.catch(err => {
console.error("Clipboard write failed:", err);
fallbackCopy(text);
});
} else {
fallbackCopy(text);
}
}

function fallbackCopy(text) {
const textarea = document.createElement("textarea");
textarea.value = text;
textarea.style.position = "fixed";
textarea.style.left = "-9999px";
document.body.appendChild(textarea);
textarea.select();
try {
document.execCommand("copy");
alert("Copied to clipboard");
} catch (err) {
console.error("Fallback copy failed:", err);
alert("Could not copy to clipboard. Please copy manually.");
} finally {
document.body.removeChild(textarea);
}
}
</script>
</details>

### D3. Define Model Metadata

Set up the input example, model name, and registration tags. Below is an example of metadata that you need to log with your model. Feel free to update them as needed:
- Input example
- Model name
- Tags

In [0]:
input_example = format_prompt(success_prompt)

model_name = "reproducible-agents-lab"

tags_to_register = {
    "framework": "openai",
    "stage": "dev",
    "version": "1"
}

### D4. Configure Resources

Define the Unity Catalog resources that your agent will use. Note this list should contain the same libraries you brought in earlier.
- Recall `agent_config` contains the tool list created earlier along with the endpoint name
- Also recall that at the start of this lab, a table was created for you called `sf_airbnb_listings`, which is what our tools' logic is based on

In [0]:
## Create a resources list with:

resources = [
    DatabricksFunction(<FILL_IN>),
    DatabricksFunction(<FILL_IN>),
    DatabricksTable(<FILL_IN>),
    DatabricksServingEndpoint(<FILL_IN>)
]

##### Task D4.1: Configure Resources ANSWER
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
resources = [
    DatabricksFunction(function_name=f"{agent_config.get('tool_list')[0]}"),
    DatabricksFunction(function_name=f"{agent_config.get('tool_list')[1]}"),
    DatabricksTable(table_name=f"{catalog_name}.{schema_name}.sf_airbnb_listings"),
    DatabricksServingEndpoint(endpoint_name=agent_config['llm_endpoint'])
]
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
const el = document.getElementById("copy-block");
if (!el) return;

const text = el.innerText;

// Preferred modern API
if (navigator.clipboard && navigator.clipboard.writeText) {
navigator.clipboard.writeText(text)
.then(() => alert("Copied to clipboard"))
.catch(err => {
console.error("Clipboard write failed:", err);
fallbackCopy(text);
});
} else {
fallbackCopy(text);
}
}

function fallbackCopy(text) {
const textarea = document.createElement("textarea");
textarea.value = text;
textarea.style.position = "fixed";
textarea.style.left = "-9999px";
document.body.appendChild(textarea);
textarea.select();
try {
document.execCommand("copy");
alert("Copied to clipboard");
} catch (err) {
console.error("Fallback copy failed:", err);
alert("Could not copy to clipboard. Please copy manually.");
} finally {
document.body.removeChild(textarea);
}
}
</script>
</details>

### D5. Load Updated Agent

Load the updated agent that includes your custom tracing functionality.

In [0]:
%reload_ext autoreload
%autoreload 2

from lab_agent_update import AGENT as updated_agent

### D6. Test Updated Agent

Verify that your updated agent works correctly before logging it. Recall we had our two variables `success_prompt_formatted` and `fail_prompt_formatted` that succeeded and failed, respectively, during our testing in section **B. Custom Tracing with Tags**.

In [0]:
updated_agent.predict(success_prompt_formatted)

In [0]:
updated_agent.predict(fail_prompt_formatted)

### D7. Log Agent to MLflow

Log your agent to MLflow with all necessary dependencies and configuration.

In [0]:
## Complete the MLflow logging process:
## - Start an MLflow run
## - Set tags using mlflow.set_tags()
## - Log the model using mlflow.pyfunc.log_model() with appropriate parameters
## - Save the model URI for later use

with mlflow.<FILL_IN>:
    <FILL_IN>
    logged_agent_info = mlflow.<FILL_IN>(
        name=<FILL_IN>,
        python_model=<FILL_IN>,
        code_paths=<FILL_IN>,
        input_example=<FILL_IN>,
        pip_requirements=[
            "databricks-openai",
            "backoff",
            f"databricks-connect=={version('databricks-connect').version}",
        ],
        resources=<FILL_IN>
    )
    model_uri = <FILL_IN>

##### Task D7.1: Log Agent to MLflow ANSWER
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
with mlflow.start_run():
    mlflow.set_tags(tags_to_register)
    logged_agent_info = mlflow.pyfunc.log_model(
        name=model_name,
        python_model="lab_agent_update.py",
        code_paths=["lab_agent_config.json"],
        input_example=input_example,
        pip_requirements=[
            "databricks-openai",
            "backoff",
            f"databricks-connect=={version('databricks-connect')}",
        ],
        resources=resources
    )
    model_uri = logged_agent_info.model_uri
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
const el = document.getElementById("copy-block");
if (!el) return;

const text = el.innerText;

// Preferred modern API
if (navigator.clipboard && navigator.clipboard.writeText) {
navigator.clipboard.writeText(text)
.then(() => alert("Copied to clipboard"))
.catch(err => {
console.error("Clipboard write failed:", err);
fallbackCopy(text);
});
} else {
fallbackCopy(text);
}
}

function fallbackCopy(text) {
const textarea = document.createElement("textarea");
textarea.value = text;
textarea.style.position = "fixed";
textarea.style.left = "-9999px";
document.body.appendChild(textarea);
textarea.select();
try {
document.execCommand("copy");
alert("Copied to clipboard");
} catch (err) {
console.error("Fallback copy failed:", err);
alert("Could not copy to clipboard. Please copy manually.");
} finally {
document.body.removeChild(textarea);
}
}
</script>
</details>

### D8. Test MLflow Model Inference

Test your logged model to ensure it works correctly.

In [0]:
## Load the model (pyfunc flavor)
## The model is logged with the input example you defined earlier
## Verify the model with the provided input data using the logged dependencies

pyfunc_model = mlflow.<FILL_IN>

input_data = pyfunc_model.<FILL_IN>

result = mlflow.<FILL_IN>(
    model_uri=<FILL_IN>,
    input_data=<FILL_IN>,
    env_manager="uv",
)

##### Task D8.1: Test MLflow Model Inference ANSWER
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
# Load the model (pyfunc flavor)
# The model is logged with the input example you defined earlier
# Verify the model with the provided input data using the logged dependencies

pyfunc_model = mlflow.pyfunc.load_model(model_uri)

input_data = pyfunc_model.input_example

result = mlflow.models.predict(
    model_uri=model_uri,
    input_data=input_data,
    env_manager="uv",
)
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
const el = document.getElementById("copy-block");
if (!el) return;

const text = el.innerText;

// Preferred modern API
if (navigator.clipboard && navigator.clipboard.writeText) {
navigator.clipboard.writeText(text)
.then(() => alert("Copied to clipboard"))
.catch(err => {
console.error("Clipboard write failed:", err);
fallbackCopy(text);
});
} else {
fallbackCopy(text);
}
}

function fallbackCopy(text) {
const textarea = document.createElement("textarea");
textarea.value = text;
textarea.style.position = "fixed";
textarea.style.left = "-9999px";
document.body.appendChild(textarea);
textarea.select();
try {
document.execCommand("copy");
alert("Copied to clipboard");
} catch (err) {
console.error("Fallback copy failed:", err);
alert("Could not copy to clipboard. Please copy manually.");
} finally {
document.body.removeChild(textarea);
}
}
</script>
</details>

### D9. Register Agent to Unity Catalog

Register your agent to Unity Catalog for governance and production deployment.

In [0]:
## Complete the Unity Catalog registration:
## - Set the registry URI to "databricks-uc"
## - Create the UC model name using catalog, schema, and model name
## - Register the model using mlflow.register_model()

mlflow.<FILL_IN>
UC_MODEL_NAME = f"{catalog_name}.{schema_name}.{model_name}"

uc_registered_model_info = mlflow.<FILL_IN>(
    model_uri=<FILL_IN>, 
    name=<FILL_IN>
)

##### Task D9.1: Register Agent to Unity Catalog ANSWER
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
mlflow.set_registry_uri("databricks-uc")
UC_MODEL_NAME = f"{catalog_name}.{schema_name}.{model_name}"

uc_registered_model_info = mlflow.register_model(
    model_uri=model_uri, 
    name=UC_MODEL_NAME
)
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
const el = document.getElementById("copy-block");
if (!el) return;

const text = el.innerText;

// Preferred modern API
if (navigator.clipboard && navigator.clipboard.writeText) {
navigator.clipboard.writeText(text)
.then(() => alert("Copied to clipboard"))
.catch(err => {
console.error("Clipboard write failed:", err);
fallbackCopy(text);
});
} else {
fallbackCopy(text);
}
}

function fallbackCopy(text) {
const textarea = document.createElement("textarea");
textarea.value = text;
textarea.style.position = "fixed";
textarea.style.left = "-9999px";
document.body.appendChild(textarea);
textarea.select();
try {
document.execCommand("copy");
alert("Copied to clipboard");
} catch (err) {
console.error("Fallback copy failed:", err);
alert("Could not copy to clipboard. Please copy manually.");
} finally {
document.body.removeChild(textarea);
}
}
</script>
</details>

### D10. Test Unity Catalog Model Inference

Test your Unity Catalog registered model to ensure it works correctly and verify that the trace appears in the UI by navigating to your model (located at `catalog_name.schema_name.reproducible-agents-lab`) and click on the most recent version of the model. There, you will find **Traces** - click on it and view the trace after running the next cell.

In [0]:
import mlflow
from mlflow.types.responses import ResponsesAgentRequest

# Load the model (pyfunc flavor)
pyfunc_model = mlflow.pyfunc.load_model(model_uri)

# The model is logged with an input example
input_data = format_prompt("What is the price average for Haight Ashbury")

# Verify the model with the provided input data using the logged dependencies
result = mlflow.models.predict(
    model_uri=model_uri,
    input_data=input_data,
    env_manager="uv",
)

## Conclusion

By completing this lab, you have successfully completed this lab on building reproducible AI agents with MLflow tracing. Throughout this hands-on exercise, you have:

- **Implemented custom tracing functions** with proper validation and error handling to monitor agent behavior
- **Applied MLflow tagging strategies** to organize and categorize traces for better observability and debugging
- **Created tool validation functions** that ensure your agents use appropriate tools and follow expected behavior patterns
- **Logged and registered AI agents** to Unity Catalog with complete dependency management for production deployment
- **Successfully registered and tested agents** from both MLflow and Unity Catalog registries

These skills are essential for building production-ready AI systems that are observable, reproducible, and governable.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>